# 06 — Comparing representations for PTAL and EPC prediction

This notebook provides the main comparison of the pretrained representations. Every feature set is evaluated with the same Ridge regression model, which offers a transparent linear measure of how readily each representation supports the two prediction tasks.

Performance is evaluated by repeatedly holding out complete groups of London boroughs. In each of five rounds, the held-out boroughs are used only for final testing. Missing-value treatment, standardisation, categorical encoding and selection of the Ridge penalty are learned from the remaining boroughs. This design estimates transfer to administrative areas that were not used to fit the model, although nearby boroughs can still share spatial context.

The principal measures are R², RMSE and MAE. R² describes the share of outcome variation captured by the predictions; RMSE and MAE express prediction error in the outcome's original units. Results are reported both as the average across the five held-out rounds and after pooling all predictions made while samples were held out.

## Main findings

For PTAL, combining all representations with Street View availability information performs best, with mean R² of approximately 0.693 and pooled R² of 0.710. The strongest individual representations are AlphaEarth (0.615), Street View CLIP (0.613) and DINOv2 (0.595), compared with 0.401 for the location-only baseline.

For EPC, the all-representation model reaches mean R² of about 0.375 when only representations are used. DINOv2 is the strongest individual representation at 0.334. SatCLIP (0.015) and Street View availability information alone (0.021) provide little predictive value for EPC.

The richer EPC control model reaches mean R² of about 0.583, but it uses floor area and construction age derived from the EPC records. It is therefore interpreted as a strong within-dataset reference rather than a like-for-like remote-observation model. Adding Street View availability information to the full representation set changes performance only slightly.

In [ ]:
# Connect Google Drive and load packages for the Ridge benchmark.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import gc
import time
import hashlib
import platform
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({
    name: getattr(_config, name)
    for name in dir(_config)
    if not name.startswith("_")
})

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

print("Python:", platform.python_version())
print("numpy:", np.__version__, "pandas:", pd.__version__, "sklearn:", sklearn.__version__)
print("Final model table:", FINAL_MODEL_TABLE_PATH, FINAL_MODEL_TABLE_PATH.exists())
print("Feature manifest:", FEATURE_MANIFEST_JSON_PATH, FEATURE_MANIFEST_JSON_PATH.exists())
print("Outer folds:", RIDGE_OUTER_SPLITS, "| Inner folds:", RIDGE_INNER_SPLITS)
print("Alpha grid:", RIDGE_ALPHA_GRID)

assert FINAL_MODEL_TABLE_PATH.exists()
assert FEATURE_MANIFEST_JSON_PATH.exists()

## 1. Load the modelling table and declared feature sets

Predictors are taken only from the manifest created in Notebook 05. Rows are ordered by task and sample ID before the borough divisions are constructed, making the evaluation independent of the physical row order in the Parquet file.

In [ ]:
# Read the canonical table and the declared feature-set manifest.
df = pd.read_parquet(FINAL_MODEL_TABLE_PATH)

with open(FEATURE_MANIFEST_JSON_PATH, "r") as f:
    manifest = json.load(f)

target_col = manifest["target_column"]
group_col = manifest["group_column"]
categorical_master = set(manifest["categorical_columns"])
feature_sets = manifest["feature_sets"]

assert len(df) == 26597
assert df["sample_id"].is_unique
assert df[target_col].notna().all()
assert df[group_col].notna().all()
assert df[group_col].nunique() == 33
assert set(df["task"].unique()) == {"PTAL", "EPC"}

df = df.sort_values(["task", "sample_id"], kind="mergesort").reset_index(drop=True)

model_key_frame = df[["sample_id", "task", group_col, target_col]].copy()
model_key_hash = hashlib.sha256(
    pd.util.hash_pandas_object(model_key_frame, index=False).values.tobytes()
).hexdigest()
manifest_hash = hashlib.sha256(
    json.dumps(manifest, sort_keys=True).encode("utf-8")
).hexdigest()

print("Loaded:", df.shape)
print("Model-key SHA256:", model_key_hash)
print("Manifest SHA256:", manifest_hash)
display(df["task"].value_counts().rename("n").to_frame())

## 2. Define the comparison

The comparison includes five individual representation families, separate Street View content and availability variants, three theory-informed compact combinations, and two all-representation combinations. The compact combinations test whether related views of place can provide most of the value of the full feature set without exhaustively searching every possible subset.

PTAL includes a location-only baseline. EPC includes a compact control baseline and the richer floor-area-and-age reference. Models that add representations to these controls are examined separately in Notebook 07 so that the representation-only ranking remains clear.

In [ ]:
# Specify the individual, compact-fusion and full-fusion comparisons for each task.
common_rep_sets = [
    "SatCLIP",
    "TESSERA",
    "AlphaEarth",
    "DINOv2",
    "StreetView_CLIP_only",
    "StreetView_metadata_only",
    "StreetView_CLIP_plus_metadata",
    "Location_and_EO",
    "Sky_and_space",
    "Street_and_sky",
    "All_representations_without_SV_metadata",
    "All_representations_plus_SV_metadata",
]

core_sets_by_task = {
    "PTAL": ["PTAL_spatial_baseline", *common_rep_sets],
    "EPC": ["EPC_controls_sparse", "EPC_controls_extensive", *common_rep_sets],
}

for task, sets in core_sets_by_task.items():
    assert len(sets) == len(set(sets)), f"{task}: duplicate feature-set names"
    missing_sets = [s for s in sets if s not in feature_sets]
    assert not missing_sets, f"{task}: missing feature sets {missing_sets}"
    for feature_set in sets:
        cols = feature_sets[feature_set]
        assert cols, f"{task} / {feature_set}: empty feature set"
        assert len(cols) == len(set(cols)), f"{task} / {feature_set}: duplicate columns"
        missing_cols = [c for c in cols if c not in df.columns]
        assert not missing_cols, f"{task} / {feature_set}: missing columns {missing_cols}"

run_spec_base = {
    "run_spec_version": "06-v2-2026-08-22",
    "notebook": "06_final_ridge_representation_benchmark.ipynb",
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "sklearn_version": sklearn.__version__,
    "target_column": target_col,
    "group_column": group_col,
    "outer_splits": int(RIDGE_OUTER_SPLITS),
    "inner_splits": int(RIDGE_INNER_SPLITS),
    "alpha_grid": [float(x) for x in RIDGE_ALPHA_GRID],
    "inner_selection_metric": "RMSE",
    "feature_sets_by_task": core_sets_by_task,
    "preprocessing": {
        "numeric_imputation": "training-fold median",
        "numeric_scaling": "training-fold StandardScaler",
        "categorical_imputation": "training-fold constant __MISSING__",
        "categorical_encoding": "training-fold OneHotEncoder(handle_unknown=ignore)",
        "streetview_clip_missing": "training-fold median",
    },
}

display(pd.DataFrame([
    {"task": task, "feature_set": fs, "n_features": len(feature_sets[fs])}
    for task, sets in core_sets_by_task.items()
    for fs in sets
]))
print("Base run specification prepared; fold hash is added and frozen in Section 5.")

## 3. Represent the absence of Street View imagery

No-image locations are a structural coverage condition rather than ordinary random missing values. Their image count is set to zero, and an unavailable distance is represented by the task's maximum search radius. Observed metadata are not overwritten. The 512 visual dimensions remain missing and are filled using only the current training data during model fitting.

In [ ]:
# Encode known Street View non-coverage while leaving visual values for training-based filling.
SV_META_COLS = [
    "sv_has_streetview", "sv_n_images", "sv_min_dist_m", "sv_mean_dist_m"
]
SV_CLIP_COLS = feature_sets["StreetView_CLIP_only"]


def apply_structural_sv_metadata_fill(task_df, task):
    out = task_df.copy()
    assert all(c in out.columns for c in SV_META_COLS)

    radius = {
        "PTAL": float(STREETVIEW_PTAL_RADIUS_M),
        "EPC": float(STREETVIEW_EPC_RADIUS_M),
    }[task]

    has = pd.to_numeric(out["sv_has_streetview"], errors="coerce")
    clip_complete = out[SV_CLIP_COLS].notna().all(axis=1)
    clip_all_missing = out[SV_CLIP_COLS].isna().all(axis=1)

    # A missing availability flag is resolved only from the already-frozen
    # image-vector presence, never from the target.
    has = has.where(has.notna(), clip_complete.astype(int)).astype(int)
    assert has.isin([0, 1]).all()
    assert clip_complete[has.eq(1)].all(), f"{task}: flagged imagery with incomplete CLIP"
    assert clip_all_missing[has.eq(0)].all(), f"{task}: no-image flag with partial CLIP vector"

    out["sv_has_streetview"] = has
    out["sv_n_images"] = pd.to_numeric(out["sv_n_images"], errors="coerce")
    out["sv_min_dist_m"] = pd.to_numeric(out["sv_min_dist_m"], errors="coerce")
    out["sv_mean_dist_m"] = pd.to_numeric(out["sv_mean_dist_m"], errors="coerce")

    no_sv = has.eq(0)
    out.loc[no_sv, "sv_n_images"] = 0.0
    for c in ["sv_min_dist_m", "sv_mean_dist_m"]:
        out.loc[no_sv & out[c].isna(), c] = radius

    assert out.loc[no_sv, "sv_n_images"].eq(0).all()
    assert out.loc[no_sv, ["sv_min_dist_m", "sv_mean_dist_m"]].notna().all().all()
    assert (out.loc[has.eq(1), "sv_n_images"].dropna() >= 1).all()
    return out


sv_audit_rows = []
for task in ["PTAL", "EPC"]:
    task_raw = df[df["task"] == task].copy()
    task_filled = apply_structural_sv_metadata_fill(task_raw, task)
    has = task_filled["sv_has_streetview"].eq(1)
    sv_audit_rows.append({
        "task": task,
        "n": int(len(task_filled)),
        "n_with_streetview": int(has.sum()),
        "coverage_pct": float(100 * has.mean()),
        "n_no_streetview": int((~has).sum()),
        "covered_rows_with_missing_metadata": int(
            task_filled.loc[has, SV_META_COLS[1:]].isna().any(axis=1).sum()
        ),
    })

sv_audit = pd.DataFrame(sv_audit_rows)
display(sv_audit)

## 4. Define the Ridge training pipeline

For every training split, numerical missing values are replaced by training medians and standardised; categorical missing values receive an explicit category and all categories are one-hot encoded. Ridge then estimates a linear relationship while shrinking coefficients to reduce instability in wide feature sets. The penalty strength is selected using three further borough-grouped divisions within the training data.

In [ ]:
# Build a preprocessing and Ridge pipeline learned entirely from training data.
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_pipeline(feature_cols):
    categorical_cols = [c for c in feature_cols if c in categorical_master]
    numeric_cols = [c for c in feature_cols if c not in categorical_master]
    transformers = []

    if numeric_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ))

    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("onehot", make_onehot()),
            ]),
            categorical_cols,
        ))

    pre = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.0,
    )
    return Pipeline([
        ("preprocess", pre),
        ("ridge", Ridge(solver="lsqr", max_iter=5000, tol=1e-4)),
    ])


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def atomic_csv(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_csv(tmp, index=False)
    tmp.replace(path)


def atomic_parquet(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)


def coerce_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    mapped = series.astype(str).str.strip().str.lower().map({"true": True, "false": False})
    assert mapped.notna().all(), "Unexpected boolean encoding."
    return mapped.astype(bool)

## 5. Create the five borough-based test rounds

All feature sets for a task use exactly the same held-out samples in each round. The saved assignments are compared with a deterministic reconstruction so that every model is evaluated on a common geographical basis.

In [ ]:
# Assign complete borough groups to five shared test rounds.
outer_assignment_rows = []
fold_index_by_task = {}

for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    groups = task_df[group_col].astype(str).to_numpy()
    assert len(np.unique(groups)) >= RIDGE_OUTER_SPLITS

    splitter = GroupKFold(n_splits=RIDGE_OUTER_SPLITS)
    task_fold = np.full(len(task_df), -1, dtype=int)

    for fold, (_, test_idx) in enumerate(splitter.split(task_df, task_df[target_col], groups)):
        task_fold[test_idx] = fold
        for idx in test_idx:
            outer_assignment_rows.append({
                "sample_id": task_df.loc[idx, "sample_id"],
                "task": task,
                "outer_fold": int(fold),
                "borough_code": task_df.loc[idx, group_col],
                "target": float(task_df.loc[idx, target_col]),
            })

    assert (task_fold >= 0).all()
    fold_index_by_task[task] = task_fold

outer_folds = (
    pd.DataFrame(outer_assignment_rows)
    .sort_values(["task", "sample_id"], kind="mergesort")
    .reset_index(drop=True)
)
assert len(outer_folds) == len(df)
assert not outer_folds.duplicated(["sample_id"]).any()

# No borough can occur in more than one outer test fold within a task.
borough_fold_counts = outer_folds.groupby(["task", "borough_code"])["outer_fold"].nunique()
assert borough_fold_counts.eq(1).all()

if RIDGE_OUTER_FOLDS_PATH.exists():
    old = pd.read_csv(RIDGE_OUTER_FOLDS_PATH).sort_values(
        ["task", "sample_id"], kind="mergesort"
    ).reset_index(drop=True)
    pd.testing.assert_frame_equal(
        old[outer_folds.columns], outer_folds,
        check_dtype=False, check_exact=False, rtol=0, atol=1e-12,
    )
    print("Verified existing frozen outer-fold assignments.")
else:
    atomic_csv(outer_folds, RIDGE_OUTER_FOLDS_PATH)
    print("Saved frozen outer-fold assignments:", RIDGE_OUTER_FOLDS_PATH)

fold_assignment_hash = hashlib.sha256(
    pd.util.hash_pandas_object(outer_folds, index=False).values.tobytes()
).hexdigest()

run_spec = {
    **run_spec_base,
    "outer_fold_assignment_sha256": fold_assignment_hash,
}
if RIDGE_RUN_SPEC_PATH.exists():
    with open(RIDGE_RUN_SPEC_PATH, "r") as f:
        existing_spec = json.load(f)
    assert existing_spec == run_spec, (
        "Existing Ridge checkpoints were created under a different run specification. "
        "Do not mix runs; archive or explicitly clear the old Notebook-06 outputs first."
    )
else:
    with open(RIDGE_RUN_SPEC_PATH, "w") as f:
        json.dump(run_spec, f, indent=2)

fold_qa = (
    outer_folds.groupby(["task", "outer_fold"])
    .agg(
        n=("sample_id", "size"),
        n_boroughs=("borough_code", "nunique"),
        target_mean=("target", "mean"),
        target_std=("target", "std"),
        target_min=("target", "min"),
        target_max=("target", "max"),
    )
    .reset_index()
)
assert fold_qa["target_std"].gt(0).all()
display(fold_qa)
print("Fold-assignment SHA256:", fold_assignment_hash)
print("Run specification frozen:", RIDGE_RUN_SPEC_PATH)

## 6. Fit the representation models

Each task–feature-set combination is fitted in five outer rounds. Results and held-out predictions are saved after each round so that a disconnected session can continue without repeating completed work. Stored results are reused only when their model definition and held-out sample IDs match the current analysis.

In [ ]:
# Fit every task and feature set, saving one result and prediction file per round.
expected_run_rows = [
    {"task": task, "feature_set": fs, "outer_fold": fold}
    for task, sets in core_sets_by_task.items()
    for fs in sets
    for fold in range(RIDGE_OUTER_SPLITS)
]
expected_runs_df = pd.DataFrame(expected_run_rows)
expected_keys = set(map(tuple, expected_runs_df[["task", "feature_set", "outer_fold"]].to_numpy()))

if RIDGE_CORE_RESULTS_PATH.exists():
    completed = pd.read_csv(RIDGE_CORE_RESULTS_PATH)
    required_result_cols = {"task", "feature_set", "outer_fold"}
    assert required_result_cols.issubset(completed.columns)
    assert not completed.duplicated(["task", "feature_set", "outer_fold"]).any(), (
        "Duplicate run keys in existing result ledger."
    )
    completed["outer_fold"] = completed["outer_fold"].astype(int)
    completed_keys = set(map(tuple, completed[["task", "feature_set", "outer_fold"]].to_numpy()))
    assert completed_keys.issubset(expected_keys), (
        "Existing result ledger contains runs outside the frozen specification."
    )
else:
    completed = pd.DataFrame()

result_rows = [] if completed.empty else completed.to_dict("records")


def checkpoint_is_valid(path, task, feature_set, outer_fold, expected_ids):
    if not path.exists():
        return False
    try:
        p = pd.read_parquet(path)
        required = {
            "sample_id", "task", "feature_set", "outer_fold",
            "borough_code", "y_true", "y_pred",
        }
        if not required.issubset(p.columns) or p["sample_id"].duplicated().any():
            return False
        if not p["task"].eq(task).all() or not p["feature_set"].eq(feature_set).all():
            return False
        if not p["outer_fold"].astype(int).eq(outer_fold).all():
            return False
        if not np.isfinite(p["y_true"]).all() or not np.isfinite(p["y_pred"]).all():
            return False
        observed_ids = set(p["sample_id"].astype(str))
        return observed_ids == set(pd.Series(expected_ids).astype(str))
    except Exception:
        return False

for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    task_df = apply_structural_sv_metadata_fill(task_df, task)
    y = pd.to_numeric(task_df[target_col], errors="raise").to_numpy()
    groups = task_df[group_col].astype(str).to_numpy()
    task_fold = fold_index_by_task[task]

    for feature_set in core_sets_by_task[task]:
        cols = feature_sets[feature_set]
        X = task_df[cols]

        for outer_fold in range(RIDGE_OUTER_SPLITS):
            test_idx = np.flatnonzero(task_fold == outer_fold)
            train_idx = np.flatnonzero(task_fold != outer_fold)
            run_key = (task, feature_set, outer_fold)
            pred_file = RIDGE_CHUNK_DIR / f"{task}__{feature_set}__fold{outer_fold}.parquet"

            completed_keys_now = {
                (r["task"], r["feature_set"], int(r["outer_fold"]))
                for r in result_rows
            }
            checkpoint_ok = checkpoint_is_valid(
                pred_file,
                task,
                feature_set,
                outer_fold,
                task_df.iloc[test_idx]["sample_id"].to_numpy(),
            )
            if run_key in completed_keys_now and checkpoint_ok:
                print("SKIP validated checkpoint:", run_key)
                continue

            print("\n" + "=" * 90)
            print(task, "|", feature_set, "| outer fold", outer_fold)
            print("=" * 90)

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            g_train = groups[train_idx]
            assert len(np.unique(g_train)) >= RIDGE_INNER_SPLITS

            inner = GroupKFold(n_splits=RIDGE_INNER_SPLITS)
            inner_splits = list(inner.split(X_train, y_train, g_train))
            for inner_train, inner_valid in inner_splits:
                assert set(g_train[inner_train]).isdisjoint(set(g_train[inner_valid]))

            pipe = build_pipeline(cols)
            search = GridSearchCV(
                estimator=pipe,
                param_grid={"ridge__alpha": RIDGE_ALPHA_GRID},
                scoring="neg_root_mean_squared_error",
                cv=inner_splits,
                refit=True,
                n_jobs=1,
                return_train_score=False,
                error_score="raise",
            )

            t0 = time.time()
            search.fit(X_train, y_train)
            elapsed_s = time.time() - t0
            pred = search.predict(X_test)
            assert len(pred) == len(test_idx)
            assert np.isfinite(pred).all()

            best_alpha = float(search.best_params_["ridge__alpha"])
            alpha_edge = best_alpha in {
                float(min(RIDGE_ALPHA_GRID)), float(max(RIDGE_ALPHA_GRID))
            }
            row = {
                "task": task,
                "feature_set": feature_set,
                "outer_fold": int(outer_fold),
                "n_features_manifest": int(len(cols)),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
                "n_train_boroughs": int(len(np.unique(g_train))),
                "n_test_boroughs": int(len(np.unique(groups[test_idx]))),
                "best_alpha": best_alpha,
                "alpha_grid_edge": bool(alpha_edge),
                "inner_best_rmse": float(-search.best_score_),
                "r2": float(r2_score(y_test, pred)),
                "rmse": rmse(y_test, pred),
                "mae": float(mean_absolute_error(y_test, pred)),
                "fit_seconds": float(elapsed_s),
            }

            pred_frame = pd.DataFrame({
                "sample_id": task_df.iloc[test_idx]["sample_id"].to_numpy(),
                "task": task,
                "feature_set": feature_set,
                "outer_fold": outer_fold,
                "borough_code": groups[test_idx],
                "y_true": y_test,
                "y_pred": pred,
            })
            atomic_parquet(pred_frame, pred_file)

            result_rows = [
                r for r in result_rows
                if (r["task"], r["feature_set"], int(r["outer_fold"])) != run_key
            ]
            result_rows.append(row)
            results_now = pd.DataFrame(result_rows).sort_values(
                ["task", "feature_set", "outer_fold"], kind="mergesort"
            )
            atomic_csv(results_now, RIDGE_CORE_RESULTS_PATH)
            print(row)

            del search, pipe, X_train, X_test, pred, pred_frame
            gc.collect()

print("Nested Ridge benchmark is complete or resumed to the current safe checkpoint.")

## 7. Assemble the held-out predictions

Before the combined files are written, every expected model round must be present and each prediction file must contain the correct task, feature set, boroughs, target values and sample IDs. This ensures that the reported summaries cannot be produced from an incomplete run.

In [ ]:
# Verify that all expected held-out predictions are present before aggregation.
results = pd.read_csv(RIDGE_CORE_RESULTS_PATH)
results["outer_fold"] = results["outer_fold"].astype(int)
assert not results.duplicated(["task", "feature_set", "outer_fold"]).any()
actual_keys = set(map(tuple, results[["task", "feature_set", "outer_fold"]].to_numpy()))

missing_run_keys = sorted(expected_keys - actual_keys)
unexpected_run_keys = sorted(actual_keys - expected_keys)
print("Completed fold-runs:", len(actual_keys), "/", len(expected_keys))
print("Missing run keys:", len(missing_run_keys))
print("Unexpected run keys:", len(unexpected_run_keys))
assert not missing_run_keys, "Benchmark incomplete: rerun Section 6."
assert not unexpected_run_keys

pred_frames = []
chunk_audit_rows = []

for task, feature_set, outer_fold in sorted(expected_keys):
    path = RIDGE_CHUNK_DIR / f"{task}__{feature_set}__fold{outer_fold}.parquet"
    assert path.exists(), f"Missing prediction chunk: {path.name}"
    p = pd.read_parquet(path)
    required = {
        "sample_id", "task", "feature_set", "outer_fold",
        "borough_code", "y_true", "y_pred",
    }
    assert required.issubset(p.columns), f"Malformed prediction chunk: {path.name}"
    assert p["task"].eq(task).all()
    assert p["feature_set"].eq(feature_set).all()
    assert p["outer_fold"].astype(int).eq(outer_fold).all()
    assert p["sample_id"].is_unique
    assert np.isfinite(p["y_true"]).all() and np.isfinite(p["y_pred"]).all()

    expected_fold = outer_folds[
        (outer_folds["task"] == task)
        & (outer_folds["outer_fold"] == outer_fold)
    ][["sample_id", "borough_code", "target"]].sort_values("sample_id").reset_index(drop=True)
    observed_fold = p[["sample_id", "borough_code", "y_true"]].sort_values("sample_id").reset_index(drop=True)
    assert expected_fold["sample_id"].equals(observed_fold["sample_id"])
    assert expected_fold["borough_code"].astype(str).equals(observed_fold["borough_code"].astype(str))
    assert np.allclose(expected_fold["target"], observed_fold["y_true"], rtol=0, atol=1e-12)

    chunk_audit_rows.append({
        "task": task,
        "feature_set": feature_set,
        "outer_fold": outer_fold,
        "n_rows": int(len(p)),
        "file": path.name,
    })
    pred_frames.append(p)

preds = pd.concat(pred_frames, ignore_index=True)
assert not preds.duplicated(["sample_id", "task", "feature_set", "outer_fold"]).any()

expected_prediction_rows = sum(
    len(df[df["task"] == task]) * len(sets)
    for task, sets in core_sets_by_task.items()
)
assert len(preds) == expected_prediction_rows
print("All exact prediction chunks validated:", len(chunk_audit_rows))
print("Validated prediction rows:", len(preds))

## 8. Summarise predictive performance

Mean and standard deviation describe variation across the five geographical test rounds. Pooled metrics use every held-out prediction once and are useful because the rounds differ in sample size and target distribution. With only five rounds, the standard deviation is descriptive; very small differences between models are not treated as definitive evidence of superiority.

In [ ]:
# Summarise mean-round and pooled held-out performance.
summary_rows = []
for (task, feature_set), g in results.groupby(["task", "feature_set"]):
    pred_g = preds[(preds["task"] == task) & (preds["feature_set"] == feature_set)]
    assert len(g) == RIDGE_OUTER_SPLITS
    assert len(pred_g) == int((df["task"] == task).sum())

    summary_rows.append({
        "task": task,
        "feature_set": feature_set,
        "n_features": int(len(feature_sets[feature_set])),
        "mean_r2": float(g["r2"].mean()),
        "sd_r2": float(g["r2"].std(ddof=1)),
        "mean_rmse": float(g["rmse"].mean()),
        "sd_rmse": float(g["rmse"].std(ddof=1)),
        "mean_mae": float(g["mae"].mean()),
        "sd_mae": float(g["mae"].std(ddof=1)),
        "pooled_r2": float(r2_score(pred_g["y_true"], pred_g["y_pred"])),
        "pooled_rmse": rmse(pred_g["y_true"], pred_g["y_pred"]),
        "pooled_mae": float(mean_absolute_error(pred_g["y_true"], pred_g["y_pred"])),
        "alpha_edge_hits": int(coerce_bool(g["alpha_grid_edge"]).sum()),
        "median_best_alpha": float(g["best_alpha"].median()),
        "total_fit_minutes": float(g["fit_seconds"].sum() / 60),
    })

summary = pd.DataFrame(summary_rows).sort_values(
    ["task", "mean_r2"], ascending=[True, False], kind="mergesort"
)
summary["rank_mean_r2"] = summary.groupby("task")["mean_r2"].rank(
    method="min", ascending=False
).astype(int)

atomic_csv(summary, RIDGE_CORE_SUMMARY_PATH)
atomic_parquet(preds, RIDGE_CORE_PREDICTIONS_PATH)
display(summary)
print("Saved:", RIDGE_CORE_SUMMARY_PATH)
print("Saved:", RIDGE_CORE_PREDICTIONS_PATH)

## 9. Confirm model convergence and save the results

The selected Ridge penalty is inspected across rounds. Repeated selection of the edge of the candidate range would indicate that the range should be widened before interpreting that model. The completed analysis contains 135 model rounds and 365,761 held-out predictions, with no repeated boundary problem.

In [ ]:
# Check penalty-range adequacy and save the completed benchmark summary.
counts = results.groupby(["task", "feature_set"]).size().rename("n_outer_folds").reset_index()
edge_counts = (
    results.assign(alpha_grid_edge=coerce_bool(results["alpha_grid_edge"]))
    .groupby(["task", "feature_set"])["alpha_grid_edge"]
    .sum().rename("edge_hits").reset_index()
)
repeated_edge = edge_counts[edge_counts["edge_hits"] >= 3].copy()

audit = {
    "run_spec_path": str(RIDGE_RUN_SPEC_PATH),
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "outer_fold_assignment_sha256": fold_assignment_hash,
    "expected_fold_runs": int(len(expected_keys)),
    "completed_fold_runs": int(len(actual_keys)),
    "all_feature_sets_have_expected_outer_folds": bool(
        counts["n_outer_folds"].eq(RIDGE_OUTER_SPLITS).all()
        and len(counts) == sum(len(v) for v in core_sets_by_task.values())
    ),
    "expected_prediction_rows": int(expected_prediction_rows),
    "actual_prediction_rows": int(len(preds)),
    "prediction_rows_match_expected": bool(len(preds) == expected_prediction_rows),
    "prediction_chunks_validated": int(len(chunk_audit_rows)),
    "n_alpha_grid_edge_hits": int(edge_counts["edge_hits"].sum()),
    "n_feature_sets_with_repeated_edge_hits": int(len(repeated_edge)),
    "alpha_grid_adequacy_pass": bool(repeated_edge.empty),
    "alpha_grid": [float(x) for x in RIDGE_ALPHA_GRID],
    "outer_splits": int(RIDGE_OUTER_SPLITS),
    "inner_splits": int(RIDGE_INNER_SPLITS),
    "spatial_group": group_col,
    "primary_validation_description": "borough-grouped nested spatial CV",
    "primary_ranking_metric": "mean outer-fold R2",
    "integrity_gate_pass": True,
    "interpretation_gate_pass": bool(repeated_edge.empty),
}

with open(RIDGE_CORE_AUDIT_PATH, "w") as f:
    json.dump(audit, f, indent=2)

display(pd.Series(audit, name="value"))
if not repeated_edge.empty:
    display(repeated_edge)
    raise RuntimeError(
        "Integrity PASS, but interpretation is on hold: at least one task/feature set "
        "selected an alpha-grid boundary in >=3 outer folds. Expand the grid in a "
        "documented rerun without changing any other design choice."
    )

print("06 Ridge benchmark — integrity gate: PASS")
print("06 Ridge benchmark — alpha adequacy / interpretation gate: PASS")

## Interpretation

The representations contain substantial information about both outcomes, but their value differs by task. PTAL benefits strongly from combining views of the built environment, while EPC is best represented by aerial DINOv2 features among the individual encoders. The richer EPC record-based controls remain stronger than representation-only models, showing that construction information recorded directly for the properties is difficult to recover fully from environmental imagery. These benchmark results provide the reference point for the control-adjusted and sensitivity comparisons reported in the accompanying notebooks.